# Matched CRNN geometry comparison v1

Control: current crop-and-stretch. Candidate: same crop, aspect-preserving resize and white padding. Identical step5,000 initialization, seed, sample stream, optimizer and2,000updates/arm; architecture/alphabet unchanged. Candidate passed a small TRAIN-only capacity gate, which does not establish generalization. The control has a warm-start advantage. Reserved475-image test stays unscored. This compares short-budget adaptation, not convergence.

Use a Colab T4. Run cells in order. This notebook uses the existing large adaptation payload plus a small new overlay; no original source datasets or completed runs are replaced. Results save in this new package's Drive runs folder. Re-run only with a new run name, or set RESUME=True for an interrupted run with last.pt. Do not launch duplicate copies of an active run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime'
print(torch.cuda.get_device_name(0), torch.__version__)
OLD = Path('/content/drive/MyDrive/4 Senior/Thesis/Manchu-CRNN-Real-Adaptation-v1')
PACKAGE = Path('/content/drive/MyDrive/4 Senior/Thesis/Manchu-CRNN-Geometry-v1')
assert (OLD/'payload.zip').is_file(), OLD
assert (PACKAGE/'overlay.zip').is_file(), PACKAGE


In [ ]:
manifest=json.loads((PACKAGE/'MANIFEST.json').read_text())
assert manifest['overlay_sha256']=='26ffc3279ee43ddb4b2cc39b631b2285dc0ad187b9c5e6e3e8acca869079059d'
assert manifest['base_payload_sha256']=='5a74b64c699d299272da8e65124a1ab7214d13f5be6e46925d9bb8d78986fec5'
stage=Path('/content/manchu_geometry_v1')
stage.mkdir(exist_ok=True)
for source, expected in [(OLD/'payload.zip',manifest['base_payload_sha256']),
                          (PACKAGE/'overlay.zip',manifest['overlay_sha256'])]:
    local=Path('/content')/('geometry_'+source.name)
    shutil.copy2(source,local)
    assert hashlib.sha256(local.read_bytes()).hexdigest()==expected
    with zipfile.ZipFile(local) as archive: archive.extractall(stage)
ROOT=stage/'research'
for relative, expected in manifest['files_sha256'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==expected
subprocess.run([sys.executable,'-m','pip','install','-q','pyarrow','Pillow','numpy'],check=True)
print(json.loads((ROOT/'data/real_adaptation_v1/PROTOCOL.json').read_text())['counts'])
print('Source/checkpoint hashes verified. Reserved test will not be scored.')


## Train both arms

Each arm: Adam1e-4, gradient clip5, batch24 (eight/domain),2,000updates, development every500, no early stopping. Both initialize from the same original adaptation step5,000 weights, not the pilot. Checkpoints include input-pipeline IDs; use real_geometry_model.load_geometry_model to reload aspect weights. CPU image decoding stays outside CUDA; image transforms/model/CTC use GPU. Candidate preprocessing currently performs an extra discarded resize to reuse the exact established ink mask/box; timing includes that overhead.


In [ ]:
RUN_TRAINING=True
RESUME=False  # True only for an interrupted run; never launch a duplicate active job.
SUFFIX='cuda_01'
env=dict(os.environ,MANCHU_GEOMETRY_RUN_ROOT=str(PACKAGE/'runs'))
if RUN_TRAINING:
    for variant in ('stretch','aspect'):
        name=f'real_geometry_v1_{variant}_{SUFFIX}'
        output=PACKAGE/'runs'/name
        if (output/'RESULT.json').exists():
            result=json.loads((output/'RESULT.json').read_text())
            assert result['completed_updates']==2000
            print('Already complete:',name)
            continue
        command=[sys.executable,str(ROOT/'experiments/compare_real_geometry_v1.py'),
                 'train','--variant',variant,'--device','cuda','--name',name]
        if RESUME and (output/'last.pt').exists(): command.append('--resume')
        subprocess.run(command,cwd=ROOT,env=env,check=True)


In [ ]:
histories={}
for variant in ('stretch','aspect'):
    path=PACKAGE/'runs'/f'real_geometry_v1_{variant}_{SUFFIX}'
    history=json.loads((path/'history.json').read_text())
    histories[variant]=history
    best=min(history,key=lambda row:row['selection_cer'])
    for label,row in [('initial',history[0]),('budget_end',history[-1]),('selected',best)]:
        print(variant,label,'step',row['step'],
              {k:{'accuracy':round(row['metrics'][k]['accuracy'],4),
                  'CER':round(row['metrics'][k]['cer'],4)} for k in ('cursive','printed','synthetic')})
    print('Selected cursive collections:',best['metrics']['cursive_collections'])
assert histories['stretch'][-1]['sample_stream_sha256']==histories['aspect'][-1]['sample_stream_sha256']
print('Matched sample order verified; reserved test not evaluated.')
